# Initialize your project folder, set simulation inputs
**Before running notebook: Create folder in your file system**\
Create a new folder for your project and put all data files you want to process in it.\
Data files are typically your exported results from Sciex Analyst saved in CSV or TXT format. Make sure you exported all columns.

**File naming conventions**\
(1) Files reflecting results from the **core method** must end with **_core.txt** or **_core.csv**\
(2) Files reflecting results from the **extended method** must end with **_extended.txt** or **extended.csv**

**Change project_folder variable in the current notebook** \
Change the variable **project_folder** in the first Code block of the notebook, so that it points to the project folder you just created.\
You can copy the path of your project folder on windows by right clicking on the folder in your file explorer and choosing the option *Copy as Path*. \
The same works on Mac OS: Control-click or right-click on the folder in Finder. Press the Option (Alt) key. Choose *Copy [foldername] as Pathname*.

**Change eis_identifier variable in the current notebook** \
Change the variable **eis_identifier** in the first Code block of the notebook. The variable is used to identify extracted internal standards (EIS)
from the Compound Names in the raw data. For data orginating from the last calibration (before June 2025) the identifier is 'IDA', for data originating from
the new calibration (after June 2025) it is 'EIS'.

**Run the current notebook** \
After you run the notebook your project folder should be populated with two new subfolders: \
(1) code_parameters, which contains three new CSV files: (i) recovery_thresholds.csv, (ii) sample_parameters.csv, (iii) simulation_parameters.csv. \
(2) processed_data, which contains one subdirectory: plots \
Make sure the directories have been created

**Change your code parameters** \
(i) Adopt the recovery threshold in recovery_thresholds.csv values to the conditions of your experiments and save the changes. \
(ii) Open sample_parameters.csv and change the sample names (column **alternative name (used in results)**) in case your internal sample naming conventions are not a good choice for sharing your resuls.
The sample names will appear in your final result tables. Sample number and sample ID are used for internal linking purposes, do not change them.\
Provide sample size (column **volume, weight, number of samples, etc.**) and sample unit (column: **unit (e.g. g/mL/sample)**) for each sample.\
Change the column **used for mdl calculation** to TRUE for all blank samples you want to use for the calculation of method detection limits (MDL). \
In case you are working with passive samplers: enter **temperature in C** and **deployment time in days**. Leave the columns blank if you are not working with passive samplers. \
Change the **dilution factor**, in case it was not one. \
Save the changes and close simulation_parameters.csv. \
(iii) Change the simulation parameters in.....

**Run data_analysis.ipynb notebook**

In [15]:
# relevant input variables
project_folder = 'rachel'

# used to identify extracted internal standards (EIS), previously known as IDA
eis_identifier = 'IDA'

# used to identify internal standards from column "Component Name" - usually you do not have to change it
standard_identifiers = 'EIS|NIS|IDA|IPS|13C|d-|d3-|d5-|18O'

In [16]:
# import necessary packages
import pandas as pd
import os

# import functions from utils.py
from utils import (read_in_data_files, get_sample_id_and_name, get_compounds_and_standards)

# display settings
pd.set_option('display.max_rows', None)  # Show all rows
pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.max_colwidth', None)  # Show full width of columns

In [17]:
# create directory for output data, thow error if it already exists
if os.path.isdir(os.path.join(project_folder, 'processed_data')):
    raise ImportError("""
        Processed Data Folder already exists in your project folder.
        The script does not create a new one.            
                      """)
os.makedirs(os.path.join(project_folder, 'processed_data'))
os.makedirs(os.path.join(project_folder, 'processed_data', 'plots'))

In [18]:
# create directory for code parameters, thow error if it already exists
if os.path.isdir(os.path.join(project_folder, 'code_parameters')):
    raise ImportError("""
        Simulation Parameters Folder already exists in your project folder.
        The script does not create a new one.            
                      """)
os.makedirs(os.path.join(project_folder, 'code_parameters'))

In [19]:
### Create csv file sample_parameters.csv for sample inputs.
# Information about available samples, LCMS Code and internal names is extracted from raw data.

# combine all data files to one common dataframe
data = read_in_data_files(project_folder)

# extract list of samples from raw data
sample_list = get_sample_id_and_name(data)
display(sample_list)

# for simplicity: get rid of standard samples in sample list
sample_list = sample_list.loc[sample_list['Sample Type'] != 'Standard', :]

# create a sample input parameter file
sample_input_data = pd.DataFrame(columns=[
    'sample id', 'alternative name (used in results)', 'volume/weight/number of samples', 'unit (e.g. g/mL/sample)', 'used for mdl calculation', 'temperature in C', 'deployment time in days', 'dilution factor',
    ],
    index=sample_list.index
    )
# fill sample id column
sample_input_data['sample id'] = sample_list['Sample ID']
# fill sample names column and get rid of 'Core' and 'Ext' ending of sample names, as Core and Extended method will be combined by the code internally.
sample_names = []
for (sample_name_core, sample_name_extended) in zip(sample_list['Sample Name Core'], sample_list['Sample Name Extended']):
    if pd.isnull(sample_name_extended):
        sample_names.append(sample_name_core[:-5])
    else:
        sample_names.append(sample_name_extended[:-4])
sample_input_data['alternative name (used in results)'] = sample_names

# set default values for remaining columns
sample_input_data['volume/weight/number of samples'] = 1
sample_input_data['used for mdl calculation'] = False
sample_input_data['unit (e.g. g/mL/sample)'] = 'sample'
sample_input_data['dilution factor'] = 1

# write dataframe to csv
sample_input_data.to_csv(path_or_buf=str(os.path.join(project_folder, 'code_parameters', 'sample_parameters.csv')), index=True)

,Sample ID,Sample Type,Sample Name Core,Sample Index Core,Sample Name Extended,Sample Index Extended
Sample Number,,,,,,
0,CS0,Standard,PFAS CS0 0ng/ml Core,1,PFAS CS0 0ng/ml Ext,93.0
1,CS0,Standard,PFAS CS0 0ng/ml Core,2,PFAS CS0 0ng/ml Ext,94.0
2,CS0,Standard,PFAS CS0 0ng/ml Core,3,PFAS CS0 0ng/ml Ext,95.0
3,CS0,Standard,PFAS CS0 0ng/ml Core,4,PFAS CS0 0ng/ml Ext,96.0
4,CS0,Standard,PFAS CS0 0ng/ml Core,5,NaN,NaN
5,CS1,Standard,PFAS CS1 0.01ng/ml Core,6,PFAS CS1 0.01ng/ml Ext,97.0
6,CS1,Standard,PFAS CS1 0.01ng/ml Core,7,PFAS CS1 0.01ng/ml Ext,98.0
7,CS1,Standard,PFAS CS1 0.01ng/ml Core,8,PFAS CS1 0.01ng/ml Ext,99.0
8,CS1,Standard,PFAS CS1 0.01ng/ml Core,9,PFAS CS1 0.01ng/ml Ext,100.0


In [20]:
### Create csv file recovery_thresholds.csv for threshold inputs for recovery rates.
# Information about available extracted internal standards (EIS), previously known as IDA, is extracted from raw data.

# extract list of compound names and standard names from raw data
compounds_msms, compounds_tof, ida_ips_msms, ida_ips_tof = get_compounds_and_standards(
    data=data, sample_list=sample_list, standard_identifiers=standard_identifiers
    )

# extract only IDAs from MSMS Channel for recovery thresholds
eis_sorted = [standard for standard in ida_ips_msms if (not pd.isnull(standard) and eis_identifier in standard)]

# create default data frame for recovery thresholds
recovery_thresholds = pd.DataFrame(columns=[
    'lower threshold for recoveries [%]', 'upper threshold for recoveries [%]'
    ],
    index=eis_sorted
    )
# input default thresholds
recovery_thresholds['lower threshold for recoveries [%]'] = 50
recovery_thresholds['upper threshold for recoveries [%]'] = 150
# write data frame to csv
recovery_thresholds.to_csv(path_or_buf=str(os.path.join(project_folder, 'code_parameters', 'recovery_thresholds.csv')), index=True)

The standard: 13C2_PFOA_TOF MS has no corresponding IDA or IPS in the default MS channel. It is ignored in the following calculations.
The standard: 18O2_PFHxS _TOF MS has no corresponding IDA or IPS in the default MS channel. It is ignored in the following calculations.


In [21]:
### Create csv file simulation_parameters.csv for inputs to the data_analysis.ipynb notebook.

# create default data frame for recovery thresholds
simulation_parameters = pd.DataFrame(columns=['Parameter Name', 'Parameter Description', 'Parameter Value'])

# Write parameters, descriptions and default values to data frame.
simulation_parameters.loc[0] = [
"EIS identifier", "Repeating substring, which is used to identify extracted internal standards (EIS), formally known as IDA, from compound name.", eis_identifier
]
simulation_parameters.loc[1] = [
"NIS identifier", "Repeating substring, which is used to identify non-extracted internal standards (NIS), formally known as IPS, from compound name.", "IPS"
]

simulation_parameters.loc[2] = [
"calibration midpoint identifier", "Repeating substring, which is used to identify calibration midpoint from sample name.", "CS6"
]

simulation_parameters.loc[3] = [
"channel selection", "For some cases, when using both core and extended method, some compounds in the TOF channels are avaialbe twice within the same sample. \n"+ \
"Set parameter to core if you want to use the channel from the core method for further caluclations. \n" +\
"Set parameter to extended if you want to use the channel from the extended method for further calculations. \n" +\
"Set parameter to average if you want to use the average of both channels for further calculations."
    , "core"
]

simulation_parameters.loc[4] = [
"IAR deviation threshold", "Maximal allowed deviation of ion abundance ratios (IAR) in %. 0 is the perfect case. 50 % is EPA threshold.", 50
]

simulation_parameters.loc[5] = [
"retention time difference threshold", "Maximal allowed retention time difference in minutes when comparing target compound to related internal standard. 0.1 minutes is EPA threshold.", 0.1
]

# set parameter to index and delete column
simulation_parameters.set_index('Parameter Name', inplace=True)

# write data frame to csv
simulation_parameters.to_csv(path_or_buf=str(os.path.join(project_folder, 'code_parameters', 'simulation_parameters.csv')), index=True)